# Bloomberg data upload

Run this on the Bloomberg computer with Bloomberg Terminal open and logged in.

Order: packages, app setup, connection test, upload test, Bloomberg test, then real data.

## 1. Packages

Run this first. If install is blocked, ask IT for these packages in Jupyter: `requests`, `pandas`, `pyarrow`, `xbbg`.

In [1]:
import importlib.util
import subprocess
import sys

def ensure_package(package_name, import_name=None):
    import_name = import_name or package_name
    if importlib.util.find_spec(import_name) is not None:
        print(f"OK: {import_name}")
        return True
    print(f"Installing {package_name}...")
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", package_name])
        print(f"OK: installed {package_name}")
        return True
    except Exception as exc:
        print(f"FAILED: could not install {package_name}: {exc}")
        return False

PACKAGE_STATUS = {
    "requests": ensure_package("requests"),
    "pandas": ensure_package("pandas"),
    "pyarrow": ensure_package("pyarrow"),
    "xbbg": ensure_package("xbbg"),
}
PACKAGE_STATUS

OK: requests
OK: pandas
OK: pyarrow
Installing xbbg...
OK: installed xbbg


{'requests': True, 'pandas': True, 'pyarrow': True, 'xbbg': True}

## 2. App setup

Set the app URL. Enter the bridge key in the prompt.

In [6]:
import getpass
import os

APP_DOMAIN = "https://rdtalpha.xyz"
BRIDGE_ID = "supervisor-bloomberg-terminal-01"

os.environ["BT_BLOOMBERG_ENDPOINT"] = APP_DOMAIN.rstrip("/")
os.environ["BT_BLOOMBERG_BRIDGE_ID"] = BRIDGE_ID
os.environ["BT_BLOOMBERG_BRIDGE_KEY"] = getpass.getpass("Bridge key: ")

ENDPOINT = os.environ["BT_BLOOMBERG_ENDPOINT"].rstrip("/")
BRIDGE_ID = os.environ["BT_BLOOMBERG_BRIDGE_ID"]
BRIDGE_KEY = os.environ["BT_BLOOMBERG_BRIDGE_KEY"]

assert ENDPOINT.startswith("https://"), "Use the public HTTPS app URL"
assert BRIDGE_KEY, "Bridge key is required"
print("Configured endpoint:", ENDPOINT)
print("Bridge id:", BRIDGE_ID)

Bridge key:  ········


Configured endpoint: https://rdtalpha.xyz
Bridge id: supervisor-bloomberg-terminal-01


## 3. Connection test

Expected result: HTTP 200.

In [8]:
import json
import requests

def bridge_headers():
    return {
        "X-Bloomberg-Bridge-Key": BRIDGE_KEY,
        "X-Bloomberg-Bridge-Id": BRIDGE_ID,
    }

response = requests.get(f"{ENDPOINT}/bridge/bloomberg/health", headers=bridge_headers(), timeout=30)
print("HTTP", response.status_code)
print(response.text[:2000])
response.raise_for_status()

HTTP 200
{"ok":true,"bridge_id":"supervisor-bloomberg-terminal-01","server_time":"2026-05-13T13:24:35.281299+00:00","accepted_formats":["parquet","jsonl","ndjson","jsonl.gz","ndjson.gz"]}


## 4. Register bridge

Run this so the app shows this computer as connected.

In [ ]:
from datetime import datetime, timezone

def register_bridge(status="online"):
    capabilities = {
        "parquet": True,
        "xbbg": False,
        "bdh": False,
        "bdib": False,
        "registered_from": "jupyter_notebook",
    }
    try:
        import pandas as pd
        from io import BytesIO
        pd.DataFrame([{"ok": 1}]).to_parquet(BytesIO(), index=False)
    except Exception as exc:
        capabilities["parquet"] = False
        capabilities["parquet_error"] = str(exc)
    try:
        from xbbg import blp as register_blp
        capabilities["xbbg"] = True
        capabilities["bdh"] = hasattr(register_blp, "bdh")
        capabilities["bdib"] = hasattr(register_blp, "bdib")
    except Exception as exc:
        capabilities["xbbg_error"] = str(exc)

    response = requests.post(
        f"{ENDPOINT}/bridge/bloomberg/heartbeat",
        headers=bridge_headers(),
        json={
            "bridge_id": BRIDGE_ID,
            "status": status,
            "capabilities": capabilities,
            "preflight": {"ok": True, "registered_at": datetime.now(timezone.utc).isoformat()},
            "active_job_id": None,
            "error_message": None,
        },
        timeout=30,
    )
    print("HTTP", response.status_code)
    print(response.text[:2000])
    response.raise_for_status()
    return response.json()

REGISTER_RESULT = register_bridge()
REGISTER_RESULT

## 5. Upload functions

Run once before any upload cells.

In [11]:
from datetime import datetime, timezone
from io import BytesIO
import hashlib
import uuid

import pandas as pd

def to_parquet_bytes(frame):
    buffer = BytesIO()
    frame.to_parquet(buffer, index=False)
    return buffer.getvalue()

def sha256_hex(payload):
    return hashlib.sha256(payload).hexdigest()

def make_request_id(source):
    stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    return f"{source}-{stamp}-{uuid.uuid4().hex[:8]}"

def upload_frame(
    frame,
    *,
    source,
    kind,
    securities,
    fields,
    start_date=None,
    end_date=None,
    periodicity=None,
    overrides=None,
    timeout=120,
):
    if frame is None or frame.empty:
        raise ValueError("Refusing to upload an empty dataframe")
    clean = frame.copy()
    clean.columns = [str(column) for column in clean.columns]
    payload = to_parquet_bytes(clean)
    request_id = make_request_id(source)
    digest = sha256_hex(payload)
    manifest = {
        "schema_version": 1,
        "bridge_id": BRIDGE_ID,
        "request_id": request_id,
        "bloomberg_source": source,
        "kind": kind,
        "securities": list(securities),
        "fields": list(fields),
        "start_date": start_date,
        "end_date": end_date,
        "periodicity": periodicity,
        "overrides": overrides or {},
        "row_count": int(len(clean)),
        "columns": [str(column) for column in clean.columns],
        "data_sha256": digest,
        "created_at": datetime.now(timezone.utc).isoformat(),
    }
    files = {"file": (f"{request_id}.parquet", payload, "application/octet-stream")}
    data = {"manifest_json": json.dumps(manifest, separators=(",", ":"))}
    response = requests.post(
        f"{ENDPOINT}/bridge/bloomberg/batches",
        headers=bridge_headers(),
        data=data,
        files=files,
        timeout=timeout,
    )
    print("HTTP", response.status_code)
    print(response.text[:3000])
    response.raise_for_status()
    return response.json()

print("Upload helpers are ready")

Upload helpers are ready


## 5. Upload test

Run this before using Bloomberg data.

In [14]:
mock_frame = pd.DataFrame(
    [
        {"date": "2026-05-01", "security": "ATW MA Equity", "field": "PX_LAST", "value": 500.0},
        {"date": "2026-05-04", "security": "ATW MA Equity", "field": "PX_LAST", "value": 502.5},
        {"date": "2026-05-05", "security": "ATW MA Equity", "field": "PX_LAST", "value": 501.0},
    ]
)
LAST_UPLOAD = upload_frame(
    mock_frame,
    source="bdh",
    kind="time_series",
    securities=["ATW MA Equity"],
    fields=["PX_LAST"],
    start_date="2026-05-01",
    end_date="2026-05-05",
    periodicity="DAILY",
)
LAST_UPLOAD

HTTP 201
{"batch":{"id":"91ce3d1a-d50e-458d-8a0c-2f7050f12796","bridge_id":"supervisor-bloomberg-terminal-01","request_id":"bdh-20260513T132444Z-cc4b3f6d","bloomberg_source":"bdh","kind":"time_series","status":"succeeded","raw_object_key":"bloomberg/raw/91ce3d1a-d50e-458d-8a0c-2f7050f12796/bdh-20260513T132444Z-cc4b3f6d.parquet","manifest_object_key":"bloomberg/raw/91ce3d1a-d50e-458d-8a0c-2f7050f12796/manifest.json","normalized_object_key":"bloomberg/normalized/91ce3d1a-d50e-458d-8a0c-2f7050f12796.parquet","filename":"bdh-20260513T132444Z-cc4b3f6d.parquet","content_type":"application/octet-stream","size_bytes":3058,"data_sha256":"9988d88438f472ff4f794dc32495167060d25aca00ffb3d74e7ce018bb6b599d","row_count":3,"series_count":1,"manifest_json":{"kind":"time_series","fields":["PX_LAST"],"columns":["date","security","field","value"],"end_date":"2026-05-05","bridge_id":"supervisor-bloomberg-terminal-01","overrides":{},"row_count":3,"created_at":"2026-05-13T13:24:44.660636Z","request_id":"bdh-

{'batch': {'id': '91ce3d1a-d50e-458d-8a0c-2f7050f12796',
  'bridge_id': 'supervisor-bloomberg-terminal-01',
  'request_id': 'bdh-20260513T132444Z-cc4b3f6d',
  'bloomberg_source': 'bdh',
  'kind': 'time_series',
  'status': 'succeeded',
  'raw_object_key': 'bloomberg/raw/91ce3d1a-d50e-458d-8a0c-2f7050f12796/bdh-20260513T132444Z-cc4b3f6d.parquet',
  'manifest_object_key': 'bloomberg/raw/91ce3d1a-d50e-458d-8a0c-2f7050f12796/manifest.json',
  'normalized_object_key': 'bloomberg/normalized/91ce3d1a-d50e-458d-8a0c-2f7050f12796.parquet',
  'filename': 'bdh-20260513T132444Z-cc4b3f6d.parquet',
  'content_type': 'application/octet-stream',
  'size_bytes': 3058,
  'data_sha256': '9988d88438f472ff4f794dc32495167060d25aca00ffb3d74e7ce018bb6b599d',
  'row_count': 3,
  'series_count': 1,
  'manifest_json': {'kind': 'time_series',
   'fields': ['PX_LAST'],
   'columns': ['date', 'security', 'field', 'value'],
   'end_date': '2026-05-05',
   'bridge_id': 'supervisor-bloomberg-terminal-01',
   'override

## 6. Bloomberg test

Bloomberg Terminal must be open and logged in.

In [17]:
try:
    from xbbg import blp
    XBBG_AVAILABLE = True
    print("OK: xbbg imported")
except Exception as exc:
    XBBG_AVAILABLE = False
    print("FAILED: xbbg import/access failed")
    print(repr(exc))

if XBBG_AVAILABLE:
    sample = blp.bdh(
        tickers=["ATW MA Equity"],
        flds=["PX_LAST"],
        start_date="2026-05-01",
        end_date="2026-05-12",
    )
    display(sample.tail())

Skipped: could not import 'blpapi': No module named 'blpapi'

## 7. Table cleanup

Run once before real data uploads.

In [20]:
OHLCV_FIELDS = ["PX_OPEN", "PX_HIGH", "PX_LOW", "PX_LAST", "VOLUME"]
print("Default OHLCV fields:", OHLCV_FIELDS)

def normalize_bdh(raw, securities=None, fields=None):
    securities = [str(value) for value in (securities or [])]
    fields = [str(value).upper() for value in (fields or [])]
    if raw is None or raw.empty:
        return pd.DataFrame(columns=["date", "security", "field", "value"])
    frame = raw.copy()
    frame.index.name = frame.index.name or "date"
    if isinstance(frame.columns, pd.MultiIndex):
        long = frame.stack(list(range(frame.columns.nlevels))).reset_index()
        date_col = long.columns[0]
        value_col = long.columns[-1]
        label_cols = list(long.columns[1:-1])
        rows = []
        for _, row in long.iterrows():
            labels = [str(row[column]) for column in label_cols]
            security = next((label for label in labels if label in securities), securities[0] if securities else (labels[0] if labels else "UNKNOWN"))
            field = next((label for label in labels if label.upper() in fields), labels[-1] if labels else (fields[0] if fields else "VALUE"))
            rows.append({"date": row[date_col], "security": security, "field": field, "value": row[value_col]})
        long = pd.DataFrame(rows)
        if long.empty:
            return pd.DataFrame(columns=["date", "security", "field", "value"])
    else:
        out = frame.reset_index()
        date_col = out.columns[0]
        if "security" not in out.columns:
            out["security"] = securities[0] if len(securities) == 1 else "UNKNOWN"
        value_cols = [column for column in out.columns if column not in {date_col, "security"}]
        long = out.melt(id_vars=[date_col, "security"], value_vars=value_cols, var_name="field", value_name="value")
        long = long.rename(columns={date_col: "date"})
    long = long[["date", "security", "field", "value"]]
    long = long.dropna(subset=["value"])
    long["date"] = pd.to_datetime(long["date"], errors="coerce").dt.date.astype(str)
    long = long[long["date"] != "NaT"]
    long["value"] = pd.to_numeric(long["value"], errors="ignore")
    return long

def normalize_bdib(raw, security, field="PX_LAST"):
    if raw is None or raw.empty:
        return pd.DataFrame(columns=["datetime", "security", "field", "value"])
    frame = raw.copy().reset_index()
    datetime_column = "time" if "time" in frame.columns else frame.columns[0]
    value_column = "close" if "close" in frame.columns else frame.columns[-1]
    long = pd.DataFrame(
        {
            "datetime": pd.to_datetime(frame[datetime_column]).astype(str),
            "security": security,
            "field": field,
            "value": pd.to_numeric(frame[value_column], errors="coerce"),
        }
    ).dropna(subset=["value"])
    return long

print("Normalization helpers are ready")

Default OHLCV fields: ['PX_OPEN', 'PX_HIGH', 'PX_LOW', 'PX_LAST', 'VOLUME']
Normalization helpers are ready


## 8. Upload one daily series

Start with one security.

In [ ]:
if not XBBG_AVAILABLE:
    raise RuntimeError("xbbg is not available in this Jupyter environment")

SECURITIES = ["ATW MA Equity"]
FIELDS = OHLCV_FIELDS.copy()
START_DATE = "2024-01-01"
END_DATE = "2026-05-12"

raw_daily = blp.bdh(
    tickers=SECURITIES,
    flds=FIELDS,
    start_date=START_DATE.replace("-", ""),
    end_date=END_DATE.replace("-", ""),
)
daily_frame = normalize_bdh(raw_daily, SECURITIES, FIELDS)

if daily_frame.empty and len(FIELDS) > 1:
    print("Combined OHLCV request returned no rows. Trying fields one by one.")
    frames = []
    working_fields = []
    for field in FIELDS:
        raw_field = blp.bdh(
            tickers=SECURITIES,
            flds=[field],
            start_date=START_DATE.replace("-", ""),
            end_date=END_DATE.replace("-", ""),
        )
        field_frame = normalize_bdh(raw_field, SECURITIES, [field])
        print(field, "rows", len(field_frame), "raw shape", None if raw_field is None else raw_field.shape)
        if not field_frame.empty:
            frames.append(field_frame)
            working_fields.append(field)
    daily_frame = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame(columns=["date", "security", "field", "value"])
    FIELDS = working_fields

if daily_frame.empty:
    raise RuntimeError("Bloomberg returned no usable daily data. Try PX_LAST only, another ticker, or another date range.")

if raw_daily is not None and not raw_daily.empty:
    display(raw_daily.tail())
display(daily_frame.tail())
print("Rows:", len(daily_frame))

LAST_UPLOAD = upload_frame(
    daily_frame,
    source="bdh",
    kind="time_series",
    securities=SECURITIES,
    fields=FIELDS,
    start_date=START_DATE,
    end_date=END_DATE,
    periodicity="DAILY",
)
LAST_UPLOAD

## 9. Check MASI tickers

Edit `MASI_SYMBOLS` if needed. This checks `MA Equity` and `MC Equity`.

In [ ]:
MASI_SYMBOLS = [
    "ATW",
    "BCP",
    "IAM",
    "BOA",
    "LHM",
]

def probe_equity_candidates(symbols, suffixes=("MA Equity", "MC Equity"), field="PX_LAST"):
    rows = []
    for symbol in symbols:
        for suffix in suffixes:
            ticker = f"{symbol} {suffix}"
            try:
                probe = blp.bdh(tickers=[ticker], flds=[field], start_date="2026-05-01", end_date="2026-05-12")
                available = probe is not None and not probe.dropna(how="all").empty
                rows.append({"symbol": symbol, "ticker": ticker, "available": bool(available), "error": None})
            except Exception as exc:
                rows.append({"symbol": symbol, "ticker": ticker, "available": False, "error": str(exc)[:300]})
    return pd.DataFrame(rows)

availability = probe_equity_candidates(MASI_SYMBOLS)
display(availability)
AVAILABLE_MASI_TICKERS = availability.loc[availability["available"], "ticker"].tolist()
AVAILABLE_MASI_TICKERS

## 10. Upload MASI daily data

Only tickers that passed the check are uploaded.

In [ ]:
if not AVAILABLE_MASI_TICKERS:
    raise RuntimeError("No MASI tickers passed the availability probe")

MASI_FIELDS = OHLCV_FIELDS.copy()
MASI_START_DATE = "2020-01-01"
MASI_END_DATE = "2026-05-12"

raw_masi = blp.bdh(
    tickers=AVAILABLE_MASI_TICKERS,
    flds=MASI_FIELDS,
    start_date=MASI_START_DATE,
    end_date=MASI_END_DATE,
)
masi_daily_frame = normalize_bdh(raw_masi, AVAILABLE_MASI_TICKERS, MASI_FIELDS)
display(masi_daily_frame.tail())
print("Rows:", len(masi_daily_frame))

LAST_UPLOAD = upload_frame(
    masi_daily_frame,
    source="bdh",
    kind="time_series",
    securities=AVAILABLE_MASI_TICKERS,
    fields=MASI_FIELDS,
    start_date=MASI_START_DATE,
    end_date=MASI_END_DATE,
    periodicity="DAILY",
    timeout=300,
)
LAST_UPLOAD

## 11. Optional intraday data

Test one ticker and one date first.

In [ ]:
INTRADAY_TICKER = "ATW MA Equity"
INTRADAY_DATE = "2026-05-12"
INTRADAY_INTERVAL_MINUTES = 60

raw_intraday = blp.bdib(ticker=INTRADAY_TICKER, dt=INTRADAY_DATE, interval=INTRADAY_INTERVAL_MINUTES)
intraday_frame = normalize_bdib(raw_intraday, INTRADAY_TICKER, field="PX_LAST")
display(intraday_frame.tail())
print("Rows:", len(intraday_frame))

if intraday_frame.empty:
    print("No intraday data returned for this ticker/date/interval. Try another date or interval.")
else:
    LAST_UPLOAD = upload_frame(
        intraday_frame,
        source="bdib",
        kind="time_series",
        securities=[INTRADAY_TICKER],
        fields=["PX_LAST"],
        start_date=INTRADAY_DATE,
        end_date=INTRADAY_DATE,
        periodicity=f"INTRADAY_{INTRADAY_INTERVAL_MINUTES}",
        timeout=300,
    )
    display(LAST_UPLOAD)

## 12. Optional BQL

Run this only if BQL works in this Jupyter setup.

In [ ]:
try:
    import bql
    service = bql.Service()
    bql_response = service.execute("get(px_last) for(['ATW MA Equity'])")
    bql_tables = [item.df().reset_index() for item in bql_response]
    bql_frame = pd.concat(bql_tables, ignore_index=True) if bql_tables else pd.DataFrame()
    display(bql_frame.head())
    if not bql_frame.empty:
        LAST_UPLOAD = upload_frame(
            bql_frame,
            source="bql",
            kind="bql_table",
            securities=["ATW MA Equity"],
            fields=["px_last"],
            timeout=300,
        )
        display(LAST_UPLOAD)
except Exception as exc:
    print("BQL probe failed or BQL is unavailable:")
    print(repr(exc))

## 13. Website job listener

Run this if you want the app page to control Bloomberg jobs. Stop the Jupyter cell to stop listening.

In [ ]:
import time
from datetime import date, timedelta

MAX_INTRADAY_BUSINESS_DAYS = 5

def bridge_api_request(method, path, **kwargs):
    headers = dict(kwargs.pop("headers", {}) or {})
    headers.update(bridge_headers())
    response = requests.request(
        method,
        f"{ENDPOINT}{path}",
        headers=headers,
        timeout=kwargs.pop("timeout", 120),
        **kwargs,
    )
    if response.status_code >= 400:
        raise RuntimeError(f"{method} {path} failed: {response.status_code} {response.text[:1000]}")
    return response.json() if response.content else None

def listener_capabilities():
    caps = {
        "pandas": pd.__version__,
        "requests": requests.__version__,
        "parquet": True,
        "xbbg": False,
        "bdh": False,
        "bdib": False,
        "bql": False,
    }
    try:
        to_parquet_bytes(pd.DataFrame([{"ok": 1}]))
    except Exception as exc:
        caps["parquet"] = False
        caps["parquet_error"] = str(exc)
    try:
        from xbbg import blp as listener_blp
        caps["xbbg"] = True
        caps["bdh"] = hasattr(listener_blp, "bdh")
        caps["bdib"] = hasattr(listener_blp, "bdib")
    except Exception as exc:
        caps["xbbg_error"] = str(exc)
    try:
        import bql  # noqa: F401
        caps["bql"] = True
    except Exception as exc:
        caps["bql_error"] = str(exc)
    return caps

def send_heartbeat(status="online", active_job_id=None, preflight=None, error_message=None):
    return bridge_api_request(
        "POST",
        "/bridge/bloomberg/heartbeat",
        json={
            "bridge_id": BRIDGE_ID,
            "status": status,
            "capabilities": listener_capabilities(),
            "preflight": preflight or {},
            "active_job_id": active_job_id,
            "error_message": error_message,
        },
    )

def post_job_status(job_id, status=None, progress=None, result=None, message=None, error_message=None):
    return bridge_api_request(
        "POST",
        f"/bridge/bloomberg/jobs/{job_id}/status",
        json={
            "status": status,
            "progress": progress or {},
            "result": result or {},
            "message": message,
            "error_message": error_message,
        },
    )

def job_fields(spec):
    fields = [str(value).strip().upper() for value in spec.get("fields", []) if str(value).strip()]
    return fields or OHLCV_FIELDS.copy()

def symbol_candidates(symbol):
    clean = str(symbol).strip().upper()
    return [f"{clean} MA Equity", f"{clean} MC Equity"] if clean else []

def job_items(spec):
    raw_candidates = spec.get("security_candidates")
    if isinstance(raw_candidates, list) and raw_candidates:
        items = []
        for item in raw_candidates:
            symbol = str(item.get("symbol") or "").strip().upper()
            candidates = [str(value).strip() for value in item.get("candidates", []) if str(value).strip()]
            if symbol and candidates:
                items.append({"symbol": symbol, "candidates": candidates})
        if items:
            return items
    securities = [str(value).strip() for value in spec.get("securities", []) if str(value).strip()]
    symbols = [str(value).strip().upper() for value in spec.get("symbols", []) if str(value).strip()]
    if securities:
        return [{"symbol": symbols[idx] if idx < len(symbols) else security, "candidates": [security]} for idx, security in enumerate(securities)]
    return [{"symbol": symbol, "candidates": symbol_candidates(symbol)} for symbol in symbols]

def parse_job_date(value, default_value):
    if not value:
        return default_value
    return date.fromisoformat(str(value)[:10])

def date_chunks(start, end, chunk_days):
    chunks = []
    current = start
    step = max(1, int(chunk_days))
    while current <= end:
        chunk_end = min(end, current + timedelta(days=step - 1))
        chunks.append((current, chunk_end))
        current = chunk_end + timedelta(days=1)
    return chunks

def normalize_bdh_job(raw, securities, fields):
    if raw is None or raw.empty:
        return pd.DataFrame(columns=["date", "security", "field", "value"])
    frame = raw.copy()
    frame.index.name = "date"
    if isinstance(frame.columns, pd.MultiIndex):
        long = frame.stack(list(range(frame.columns.nlevels))).reset_index()
        long.columns = ["date", *[f"level_{idx}" for idx in range(frame.columns.nlevels)], "value"]
        rows = []
        for _, row in long.iterrows():
            labels = [str(row[f"level_{idx}"]) for idx in range(frame.columns.nlevels)]
            security = next((label for label in labels if label in securities), securities[0])
            field = next((label for label in labels if label in fields), labels[-1])
            rows.append({"date": pd.to_datetime(row["date"]).date().isoformat(), "security": security, "field": field, "value": row["value"]})
        if not rows:
            return pd.DataFrame(columns=["date", "security", "field", "value"])
        return pd.DataFrame(rows).dropna(subset=["value"])
    out = frame.reset_index()
    out["date"] = pd.to_datetime(out["date"]).dt.date.astype(str)
    if len(securities) == 1:
        out["security"] = securities[0]
    return out.melt(id_vars=["date", "security"], var_name="field", value_name="value").dropna(subset=["value"])

BDIB_FIELD_MAP = {
    "open": "PX_OPEN",
    "high": "PX_HIGH",
    "low": "PX_LOW",
    "close": "PX_LAST",
    "last": "PX_LAST",
    "volume": "VOLUME",
}

def normalize_bdib_job(raw, security, fields):
    if raw is None or raw.empty:
        return pd.DataFrame(columns=["datetime", "security", "field", "value"])
    frame = raw.copy().reset_index()
    lower_cols = {str(column).strip().lower(): str(column) for column in frame.columns}
    time_col = next((lower_cols[name] for name in ("datetime", "time", "timestamp", "index") if name in lower_cols), str(frame.columns[0]))
    requested = {field.upper() for field in fields}
    rows = []
    for raw_name, bloomberg_field in BDIB_FIELD_MAP.items():
        if requested and bloomberg_field not in requested:
            continue
        column = lower_cols.get(raw_name)
        if column is None:
            continue
        for _, row in frame[[time_col, column]].dropna(subset=[time_col, column]).iterrows():
            rows.append({"datetime": pd.to_datetime(row[time_col]).isoformat(), "security": security, "field": bloomberg_field, "value": row[column]})
    return pd.DataFrame(rows)

def fetch_bdh_job(security, fields, start, end):
    from xbbg import blp as listener_blp
    raw = listener_blp.bdh(tickers=[security], flds=fields, start_date=start.strftime("%Y%m%d"), end_date=end.strftime("%Y%m%d"))
    frame = normalize_bdh_job(raw, [security], fields)
    if not frame.empty or len(fields) <= 1:
        return frame
    frames = []
    for field in fields:
        raw_field = listener_blp.bdh(tickers=[security], flds=[field], start_date=start.strftime("%Y%m%d"), end_date=end.strftime("%Y%m%d"))
        field_frame = normalize_bdh_job(raw_field, [security], [field])
        if not field_frame.empty:
            frames.append(field_frame)
    return pd.concat(frames, ignore_index=True) if frames else frame

def fetch_bdib_job(security, fields, day, interval_minutes):
    from xbbg import blp as listener_blp
    try:
        raw = listener_blp.bdib(ticker=security, dt=day.isoformat(), interval=interval_minutes)
    except TypeError:
        raw = listener_blp.bdib(security, day.isoformat(), interval=interval_minutes)
    return normalize_bdib_job(raw, security, fields)

def probe_job_security(spec, security):
    fields = job_fields(spec)
    frequency = str(spec.get("frequency") or "daily").lower()
    probe_days = int((spec.get("options") or {}).get("probe_days") or 5)
    end = date.today()
    start = end - timedelta(days=max(2, probe_days * 2))
    try:
        if frequency == "daily":
            frame = fetch_bdh_job(security, fields[: min(len(fields), 4)], start, end)
        else:
            interval = 60 if frequency == "hourly" else 1
            frame = pd.DataFrame()
            for day in pd.date_range(start=start, end=end, freq="B")[-probe_days:]:
                frame = fetch_bdib_job(security, fields, day.date(), interval)
                if not frame.empty:
                    break
        return {"security": security, "available": not frame.empty, "rows": int(len(frame)), "start": start.isoformat(), "end": end.isoformat(), "error": None}
    except Exception as exc:
        return {"security": security, "available": False, "rows": 0, "start": start.isoformat(), "end": end.isoformat(), "error": str(exc)[:1000]}

def run_preflight_job(spec):
    result = {"endpoint": {}, "capabilities": listener_capabilities(), "probe": None}
    result["endpoint"] = bridge_api_request("GET", "/bridge/bloomberg/health")
    items = job_items(spec)
    probe_security = items[0]["candidates"][0] if items else "ATW MA Equity"
    result["probe"] = probe_job_security({**spec, "frequency": "daily", "fields": ["PX_LAST"]}, probe_security)
    result["ok"] = bool(result["endpoint"].get("ok")) and bool(result["capabilities"].get("parquet"))
    return result

def run_discovery_job(job_id, spec):
    items = job_items(spec)
    results = []
    available_count = 0
    for index, item in enumerate(items, start=1):
        chosen = None
        probes = []
        for candidate in item["candidates"]:
            probe = probe_job_security(spec, candidate)
            probes.append(probe)
            if probe["available"]:
                chosen = probe
                break
        if chosen:
            available_count += 1
        results.append({"symbol": item["symbol"], "available": chosen is not None, "selected_security": chosen["security"] if chosen else None, "probes": probes})
        post_job_status(job_id, status="running", progress={"stage": "discovery", "symbols_done": index, "symbols_total": len(items), "latest_symbol": item["symbol"]}, message=f"Discovery checked {item['symbol']}")
    return {"frequency": spec.get("frequency"), "fields": job_fields(spec), "symbols_total": len(items), "available_count": available_count, "unavailable_count": max(0, len(items) - available_count), "items": results}

def run_backfill_job(job_id, spec):
    items = job_items(spec)
    fields = job_fields(spec)
    frequency = str(spec.get("frequency") or "daily").lower()
    options = spec.get("options") or {}
    start = parse_job_date(spec.get("start_date"), date(2010, 1, 1))
    end = parse_job_date(spec.get("end_date"), date.today())
    uploaded_batches = 0
    uploaded_rows = 0
    uploaded = []
    failures = []
    for item_index, item in enumerate(items, start=1):
        selected_security = None
        for candidate in item["candidates"]:
            probe = probe_job_security({**spec, "frequency": frequency}, candidate)
            if probe["available"]:
                selected_security = candidate
                break
        if not selected_security:
            failures.append({"symbol": item["symbol"], "error": "No available Bloomberg security candidate"})
            continue
        try:
            if frequency == "daily":
                chunks = date_chunks(start, end, int(options.get("daily_chunk_days") or 365))
                for chunk_index, (chunk_start, chunk_end) in enumerate(chunks, start=1):
                    frame = fetch_bdh_job(selected_security, fields, chunk_start, chunk_end)
                    if frame.empty:
                        continue
                    upload = upload_frame(frame, source="bdh", kind="time_series", securities=[selected_security], fields=fields, start_date=chunk_start.isoformat(), end_date=chunk_end.isoformat(), periodicity="DAILY", timeout=300)
                    uploaded_batches += 1
                    uploaded_rows += int(len(frame))
                    uploaded.append({"symbol": item["symbol"], "security": selected_security, "batch": upload.get("batch", {})})
                    post_job_status(job_id, status="running", progress={"stage": "backfill", "symbols_done": item_index - 1, "symbols_total": len(items), "latest_symbol": item["symbol"], "chunk": chunk_index, "chunks": len(chunks), "uploaded_batches": uploaded_batches}, message=f"Uploaded {item['symbol']} {chunk_start} to {chunk_end}")
            else:
                interval = 60 if frequency == "hourly" else 1
                days = [day.date() for day in pd.date_range(start=start, end=end, freq="B")]
                if len(days) > MAX_INTRADAY_BUSINESS_DAYS:
                    raise RuntimeError(f"Intraday jobs are limited to {MAX_INTRADAY_BUSINESS_DAYS} business days in this notebook listener. Reduce the date range.")
                for day_index, day in enumerate(days, start=1):
                    frame = fetch_bdib_job(selected_security, fields, day, interval)
                    if frame.empty:
                        continue
                    upload = upload_frame(frame, source="bdib", kind="time_series", securities=[selected_security], fields=fields, start_date=day.isoformat(), end_date=day.isoformat(), periodicity=f"INTRADAY_{interval}", timeout=300)
                    uploaded_batches += 1
                    uploaded_rows += int(len(frame))
                    uploaded.append({"symbol": item["symbol"], "security": selected_security, "batch": upload.get("batch", {})})
                    post_job_status(job_id, status="running", progress={"stage": "backfill", "symbols_done": item_index - 1, "symbols_total": len(items), "latest_symbol": item["symbol"], "day": day.isoformat(), "days": len(days), "uploaded_batches": uploaded_batches}, message=f"Uploaded {item['symbol']} intraday through {day}")
        except Exception as exc:
            failures.append({"symbol": item["symbol"], "security": selected_security, "error": str(exc)[:1000]})
    return {"frequency": frequency, "fields": fields, "start_date": start.isoformat(), "end_date": end.isoformat(), "uploaded_batches": uploaded_batches, "uploaded_rows": uploaded_rows, "uploaded": uploaded[-25:], "failures": failures}

def execute_website_job(job):
    job_id = str(job["id"])
    spec = job.get("spec_json") or {}
    job_type = str(job.get("job_type") or spec.get("job_type") or "")
    print(f"Running job {job_id}: {job_type}")
    post_job_status(job_id, status="running", progress={"stage": "starting"}, message="Notebook bridge started job")
    send_heartbeat(status="busy", active_job_id=job_id)
    try:
        if job_type == "preflight":
            result = run_preflight_job(spec)
        elif job_type == "discovery":
            result = run_discovery_job(job_id, spec)
            if str(spec.get("mode") or "") == "discover_then_backfill":
                result = {**result, "backfill": run_backfill_job(job_id, spec)}
        elif job_type in {"backfill", "refresh"}:
            result = run_backfill_job(job_id, spec)
        else:
            raise RuntimeError(f"Unsupported job type: {job_type}")
        post_job_status(job_id, status="succeeded", progress={"stage": "completed"}, result=result, message="Notebook bridge completed job")
        send_heartbeat(status="online", preflight=result if job_type == "preflight" else None)
        print(f"Completed job {job_id}")
    except Exception as exc:
        post_job_status(job_id, status="failed", progress={"stage": "failed"}, error_message=str(exc), message="Notebook bridge job failed")
        send_heartbeat(status="online", error_message=str(exc))
        print(f"Failed job {job_id}: {exc}")

def listen_for_website_jobs(poll_seconds=10, lease_seconds=300, once=False):
    send_heartbeat(status="online")
    print(f"Listening as {BRIDGE_ID}. Stop this cell to disconnect.")
    while True:
        claim = bridge_api_request("GET", "/bridge/bloomberg/jobs/next", params={"bridge_id": BRIDGE_ID, "lease_seconds": lease_seconds})
        job = (claim or {}).get("job")
        if job:
            execute_website_job(job)
        else:
            send_heartbeat(status="online")
            print("No job", datetime.now().strftime("%H:%M:%S"))
        if once:
            break
        time.sleep(poll_seconds)

print("Website job listener is ready")

## 14. Start website job listener

Run this cell, then use the Bloomberg tab in the app. Stop this cell when done.

In [ ]:
listen_for_website_jobs(poll_seconds=10, lease_seconds=300, once=False)

## Troubleshooting

- HTTP 401 or 403: wrong bridge key.
- Timeout: network or domain block.
- `xbbg` fails: Bloomberg Python access is missing.
- Empty data: ticker, field, date range, or entitlement issue.
- Large upload fails: use fewer tickers or a shorter date range.
- Queued app jobs do not run: start the website job listener cell.